# 🛡️ Master End-to-End Deepfake Detection Pipeline
### Dual-Stream Spatial-Frequency Detector (ConvNeXt + 2D FFT) with Temperature Calibration, Sub-Domain Evaluation, Robustness Stress Testing & True LOTO Experiments

This notebook contains the complete, reproducible end-to-end research pipeline for training, calibrating, evaluating, ablating, and benchmarking the Dual-Stream Deepfake Detector.

## 🛠️ Step 1: Repository Clone & Environment Setup

In [ ]:
import os, sys, subprocess, torch

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Count: {torch.cuda.device_count()}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

if not os.path.exists('/kaggle/working/repo'):
    subprocess.run(["git", "clone", "https://github.com/yyouretoast/deepfake-detection.git", "/kaggle/working/repo"], check=True)
else:
    os.chdir('/kaggle/working/repo')
    subprocess.run(["git", "pull"], check=True)

os.chdir('/kaggle/working/repo')
if '/kaggle/working/repo' not in sys.path:
    sys.path.insert(0, '/kaggle/working/repo')
print("✅ Setup complete. Current working directory:", os.getcwd())

## 📊 Step 2: Primary Dual-Stream Model Training (5 Epochs DDP)

In [ ]:
!accelerate launch --mixed_precision fp16 --num_processes 2 --multi_gpu scripts/train_dual_stream_ddp.py

## 🌡️ Step 3: Test Set Evaluation & Log-Temperature Scaling ($T^*$) Calibration

In [ ]:
!python scripts/evaluate_test_set.py

## 🔬 Step 4: Per-Generator Sub-Domain Evaluation (2-Class AUC vs Real Faces)

In [ ]:
!python scripts/evaluate_subdomain_breakdown.py

## 🛡️ Step 5: True LOTO Experiment — Fold 1 (`NeuralTextures` Within-Dataset LOTO)

In [ ]:
!accelerate launch --mixed_precision fp16 --num_processes 2 --multi_gpu scripts/train_loto_experiment.py --holdout neuraltextures --epochs 3

## 🛡️ Step 6: True LOTO Experiment — Fold 2 (`Celeb-DF v2` Cross-Dataset LOTO)

In [ ]:
!accelerate launch --mixed_precision fp16 --num_processes 2 --multi_gpu scripts/train_loto_experiment.py --holdout celeb --epochs 3

## 🧪 Step 7: Perturbation Robustness Stress Testing (JPEG, Blur, Noise, Downscaling)

In [ ]:
!python scripts/evaluate_robustness.py

## 📸 Step 8: Dual-Stream SRM & Grad-CAM Attention Heatmap Visualization

In [ ]:
!python scripts/visualize_attention_maps.py

## 📈 Step 9: Publication Plot Generation & Master Summary

In [ ]:
!python scripts/generate_benchmark_plots.py

from IPython.display import Image, display
if os.path.exists('docs/figures/subdomain_roc_curves.png'):
    display(Image(filename='docs/figures/subdomain_roc_curves.png'))
if os.path.exists('docs/figures/calibration_reliability_diagram.png'):
    display(Image(filename='docs/figures/calibration_reliability_diagram.png'))